# PyRestore 01 — Single-scene framework demonstration (offline)

**Claim under demonstration:** one shared data & execution contract turns a georeferenced
image manifest into an analysis-ready, provenance-recorded, GIS-exportable dataset —
without the pipeline knowing anything about the task semantics.

This notebook runs fully offline: the CV and VLM indicator tables for the 12 demo panoramas
(Barcelona sample) are precomputed and shipped under `data/demo/`. The live VLM path is shown
at the end and only needs an API key in `.env`.

In [1]:
from pathlib import Path

import pandas as pd

from pyrestore import load_config, load_manifest, load_task, run_task

DEMO = Path("data/demo")

## 1. The task is a definition, not code

The PRS-11 restorative-quality task is a YAML file: prompt template, output-field contract, derived fields, validation mapping and the spatial screening column. Nothing in `pyrestore/` mentions ART or PRS.

In [2]:
task = load_task("tasks/prs11.yaml")
print("task id:", task["id"], "| signature:", task["input_signature"])
print("output fields:", {k: v["type"] for k, v in task["output_fields"].items()})
print("derived:", list(task.get("derived_fields", {})))
print("prompt template (first 3 lines):")
print("\n".join(task["prompt"]["_text"].splitlines()[:3]))

task id: restorative_quality_prs11 | signature: single_scene
output fields: {'being_away': 'score', 'coherence': 'score', 'scope': 'score', 'fascination': 'score'}
derived: ['restorative_average']
prompt template (first 3 lines):
You are an expert in environmental psychology assessing the psychological restorative
quality of an urban street-view image, grounded in Attention Restoration Theory (ART)
and the Perceived Restorativeness Scale (PRS-11).


## 2. The manifest is the spatial-identity contract

Required: `site_id, capture_id, image_path, lon, lat`; optional provenance: `captured_at, heading, camera, source, licence`. The demo manifest is a legacy PyRestore four-column file (`image, image_path, lon, lat`) — `load_manifest` adapts it automatically.

In [3]:
manifest = load_manifest(DEMO / "manifest.csv")   # legacy format adapted on load
print(f"{len(manifest)} captures; site_id == capture_id adapted from legacy 'image' column")
manifest[["site_id", "capture_id", "lon", "lat"]].head()

12 captures; site_id == capture_id adapted from legacy 'image' column


,site_id,capture_id,lon,lat
0,bcn-0029,bcn-0029,2.163503,41.396256
1,bcn-0039,bcn-0039,2.168929,41.396504
2,bcn-0084,bcn-0084,2.159168,41.395279
3,bcn-0131,bcn-0131,2.153424,41.393759
4,bcn-0144,bcn-0144,2.167262,41.393963


## 3. One call: two evidence families -> one dataset

`run_task` coordinates the CV channel (physical measurements), the VLM channel (semantic interpretations), failure-aware harmonisation, optional validation, spatial screening and export. With precomputed tables nothing calls an API.

In [4]:
cfg = load_config({"vlm": {"cache_db": "cache/vlm_cache.sqlite", "offline": True}})

result = run_task(
    DEMO / "manifest.csv",
    "tasks/prs11.yaml",
    cfg,
    cv_features=DEMO / "cv_features.csv",      # precomputed physical measurements
    vlm_scores=DEMO / "vlm_scores.csv",        # precomputed semantic interpretations
)
print("validation_status:", result["validation_status"])
result["quality"]

validation_status: not_validated


,n_input_rows,n_valid_coordinates,n_cv_success,n_cv_error,n_vlm_success,n_vlm_error
0,12,12,12,0,12,0


In [5]:
indicators = result["indicators"]
cols = ["capture_id", "lon", "lat", "GVI", "SVF", "enclosure",
        "vlm_being_away", "vlm_coherence", "vlm_scope", "vlm_fascination", "vlm_restorative_average"]
indicators[cols].head()

,capture_id,lon,lat,GVI,SVF,enclosure,vlm_being_away,vlm_coherence,vlm_scope,vlm_fascination,vlm_restorative_average
0,bcn-0029,2.163503,41.396256,0.268881,0.000000,0.243789,0.14,0.68,0.46,0.47,0.4375
1,bcn-0039,2.168929,41.396504,0.222508,0.000000,0.240079,0.18,0.62,0.46,0.33,0.3975
2,bcn-0084,2.159168,41.395279,0.000000,0.003479,0.688167,0.05,0.72,0.31,0.38,0.3650
3,bcn-0131,2.153424,41.393759,0.002821,0.000000,0.554744,0.18,0.72,0.41,0.36,0.4175
4,bcn-0144,2.167262,41.393963,0.007517,0.017347,0.536743,0.08,0.72,0.38,0.22,0.3500


## 4. Spatial screening: candidates for follow-up, not facts

Local Moran's I flags significant Low-Low clusters as *candidate* low-score areas. Every global statistic is reported with its permutation p-value; the map is a screening tool for follow-up investigation.

In [6]:
stats = pd.read_csv(result["paths"]["spatial_stats"])
print(stats.to_string(index=False))
indicators["lisa_label"].value_counts()

              value_col  n_valid  k p_adjust  global_moran_i  global_moran_p    screening         candidate_col  n_candidate_low_clusters  n_candidate_high_clusters
vlm_restorative_average       12  8   fdr_bh        0.040646           0.021 lisa_low_low candidate_low_cluster                         0                          0


lisa_label
Not significant    12
Name: count, dtype: int64

In [7]:
from IPython.display import HTML

HTML(filename=result["paths"]["map"])

## 5. Products on disk

Every run exports the same product family — observations, per-analyzer tables, the GeoPackage/CSV dataset, quality, validation and provenance records.

In [8]:
for key, path in result["paths"].items():
    if isinstance(path, list):
        shown = [str(Path(p).relative_to(Path.cwd())) for p in path]
    else:
        shown = str(Path(path).relative_to(Path.cwd()))
    print(f"{key:12s} {shown}")
provenance = pd.read_csv(result["paths"]["provenance"])
provenance.T

observations outputs/restorative_quality_prs11/observations.csv
cv_features  outputs/restorative_quality_prs11/cv_features.csv
vlm_scores   outputs/restorative_quality_prs11/vlm_scores.csv
datasets     ['outputs/restorative_quality_prs11/indicators.gpkg', 'outputs/restorative_quality_prs11/indicators.csv']
quality      outputs/restorative_quality_prs11/quality_report.csv
validation   outputs/restorative_quality_prs11/validation_report.csv
provenance   outputs/restorative_quality_prs11/provenance.csv
map          outputs/restorative_quality_prs11/screening_map.html
spatial_stats outputs/restorative_quality_prs11/spatial_stats.csv
run_manifest outputs/restorative_quality_prs11/run_manifest.json


,0
task_id,restorative_quality_prs11
task_version,1.1
input_signature,single_scene
vlm_model,gpt-5.4-mini
prompt_hash,35128c8cfbadece8
configured_vlm_model,gpt-5.4-mini
task_prompt_hash,8e3bf22c99fcf485
request_hash,2ab1f9bc6b8a5cda
precomputed_contract_match,False
cache_db,NaN


## 6. The live VLM path

Remove the precomputed tables and supply credentials in `.env` (see `.env.example`) to score
images live; responses are cached, so re-runs are free and reproducible:

```python
live = run_task(DEMO / "manifest.csv", "tasks/prs11.yaml", "config.yaml")
```

**What this notebook does not claim:** VLM scores are perceptual interpretations, not measured
human experience; the map supports preliminary visual screening, not health outcomes.